# Notebook 01 — ISIC 2019 Dataset Provenance and Split Audit

**Purpose:** Audit ISIC 2019 benchmark provenance, verify the seven-class mapping (`AK + SCC -> AKIEC`), verify frozen partition counts (`16,702 / 3,542 / 1,401 / 3,686`), explain operational grouping (`lesion_id` where available, `image_id` singleton otherwise), test split disjointness when raw metadata are present, and document `CC-BY-NC 4.0` licensing.  
**Data & Artifact Requirement:** Tracked `data/isic2019/splits/split_metadata.json` is sufficient for aggregate split verification. Raw row-level overlap and metadata completeness checks require local `ISIC_2019_Training_Metadata.csv`, `ISIC_2019_Training_GroundTruth.csv`, and `data/isic2019/splits/*_indices.csv` (`REQUIRES_DATA`); missing files are skipped explicitly without synthesizing records.

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Verify frozen ISIC 2019 split counts (`Train = 16,702`, `Val = 3,542`, `Cal = 1,401`, `Test = 3,686`, `Total = 25,331`) and `split_hash = df52266ea5c2a4b1`.
2. Distinguish **operational grouping units** (`lesion_id` where present + `image_id` singleton when missing) from true **patient-level independence** (which is **unverified** because ISIC 2019 omits `patient_id`).
3. Inspect the 7-class benchmark mapping (`MEL`, `NV`, `BCC`, `AKIEC`, `BKL`, `DF`, `VASC`) and verify zero cross-split `image_id` or `lesion_id` leakage when local CSVs are supplied.

### Prerequisites
- Completed environment check in [`00_Project_Overview_and_Quickstart.ipynb`](00_Project_Overview_and_Quickstart.ipynb).
- For row-level overlap verification only: official `CC-BY-NC 4.0` ISIC 2019 CSVs placed locally under `data/isic2019/raw/` (never downloaded automatically by this notebook).

## 2. Environment & Dataset Availability Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb01_data_split"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_META_PATH = REPO_ROOT / "data" / "isic2019" / "splits" / "split_metadata.json"
RAW_META_PATH = REPO_ROOT / "data" / "isic2019" / "raw" / "ISIC_2019_Training_Metadata.csv"
RAW_GT_PATH = REPO_ROOT / "data" / "isic2019" / "raw" / "ISIC_2019_Training_GroundTruth.csv"
SPLIT_CSVS = {
    "train": REPO_ROOT / "data" / "isic2019" / "splits" / "train_indices.csv",
    "val": REPO_ROOT / "data" / "isic2019" / "splits" / "val_indices.csv",
    "cal": REPO_ROOT / "data" / "isic2019" / "splits" / "cal_indices.csv",
    "test": REPO_ROOT / "data" / "isic2019" / "splits" / "test_indices.csv",
}

HAS_RAW_META = RAW_META_PATH.is_file() and RAW_GT_PATH.is_file()
HAS_SPLIT_CSVS = all(p.is_file() for p in SPLIT_CSVS.values())

preflight_df = pd.DataFrame([
    {"artifact": "split_metadata.json (tracked)", "path": SPLIT_META_PATH.relative_to(REPO_ROOT).as_posix(), "available": SPLIT_META_PATH.is_file()},
    {"artifact": "ISIC_2019_Training_Metadata.csv", "path": RAW_META_PATH.relative_to(REPO_ROOT).as_posix(), "available": RAW_META_PATH.is_file()},
    {"artifact": "ISIC_2019_Training_GroundTruth.csv", "path": RAW_GT_PATH.relative_to(REPO_ROOT).as_posix(), "available": RAW_GT_PATH.is_file()},
    {"artifact": "Split index CSVs (train/val/cal/test)", "path": "data/isic2019/splits/*_indices.csv", "available": HAS_SPLIT_CSVS},
])
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
display(preflight_df)

## 3. Methodology: Partition Design, Class Mapping & Operational Grouping

- **Canonical 7-Class Formulation:** ISIC 2019 Training GroundTruth lists 8 diagnostic columns (`MEL`, `NV`, `BCC`, `AK`, `BKL`, `DF`, `VASC`, `SCC`) plus zero-count `UNK`. DERMA-ACT V2 merges `AK` (`867`) + `SCC` (`628`) into `AKIEC` (`1,495`), matching the 7-class order in `research/derma_act/config.py`.
- **Operational Grouping Rule:** Because ISIC 2019 lacks a `patient_id` field, grouping assigns each image to its observed `lesion_id` where non-null (`23,247` images → `11,847` unique `lesion_id` clusters) and falls back to an `image_id` singleton token (`IMG::<image_id>`) where `lesion_id` is missing (`2,084` images), producing `13,931` operational grouping units (`seed = 42`, `split_hash = df52266ea5c2a4b1`).
- **Calibration Sub-Split:** For leak-free split-conformal calibration, `Cal` (`1,401` images, `789` groups) is partitioned disjointly by group into $\mathcal{D}_{\mathrm{fit}}$ (`698` images, `393` groups, for temperature scaling $T_{\mathrm{fit}}$) and $\mathcal{D}_{\mathrm{conf}}$ (`703` images, `396` groups, for conformal quantiles).

In [ ]:
from research.derma_act.config import CLASS_NAMES, DISPLAY_CLASSES

split_meta = json.loads(SPLIT_META_PATH.read_text(encoding="utf-8"))
expected = {"train_samples": 16702, "val_samples": 3542, "cal_samples": 1401, "test_samples": 3686, "total_samples": 25331, "num_unique_groups": 13931}
for k, v in expected.items():
    assert split_meta[k] == v, f"Mismatch on {k}: {split_meta[k]} != {v}"

split_summary = pd.DataFrame([
    {"split": "Train", "images_N": split_meta["train_samples"], "groups_K": split_meta["train_groups"], "share_pct": 100 * split_meta["train_samples"] / split_meta["total_samples"]},
    {"split": "Validation", "images_N": split_meta["val_samples"], "groups_K": split_meta["val_groups"], "share_pct": 100 * split_meta["val_samples"] / split_meta["total_samples"]},
    {"split": "Calibration", "images_N": split_meta["cal_samples"], "groups_K": split_meta["cal_groups"], "share_pct": 100 * split_meta["cal_samples"] / split_meta["total_samples"]},
    {"split": "Held-Out Test", "images_N": split_meta["test_samples"], "groups_K": split_meta["test_groups"], "share_pct": 100 * split_meta["test_samples"] / split_meta["total_samples"]},
    {"split": "Total ISIC 2019", "images_N": split_meta["total_samples"], "groups_K": split_meta["num_unique_groups"], "share_pct": 100.0},
])
print(f"Verified split_hash = {split_meta['split_hash']} | seed = {split_meta['config']['seed']} | group_column = {split_meta['group_column_used']}")
print("Canonical class mapping:", {i: f"{c.upper()} ({DISPLAY_CLASSES[c]})" for i, c in enumerate(CLASS_NAMES)})
display(split_summary)

## 4. Row-Level Metadata & Cross-Split Disjointness Audit (`REQUIRES_DATA`)

When the user supplies the official ISIC 2019 CSVs and split index files locally, the cell below verifies:
1. Exact image counts, observed `lesion_id` cluster counts, and missing-ID singleton counts per split.
2. **Zero pairwise `image_id` overlap** and **zero pairwise observed `lesion_id` overlap** across Train, Val, Cal, and Test.
3. Exact 8-column-to-7-class reconciliation (`AK + SCC = AKIEC`, `UNK = 0`).

If the local CSVs are absent (e.g., in a clean public clone), this cell skips explicitly without fabricating synthetic rows.

In [ ]:
raw_audit_df = None
overlap_df = None
class_split_df = None

if HAS_RAW_META and HAS_SPLIT_CSVS:
    meta_df = pd.read_csv(RAW_META_PATH)
    gt_df = pd.read_csv(RAW_GT_PATH)
    assert len(meta_df) == 25331 and len(gt_df) == 25331

    # Canonical 7-class mapping from raw ISIC 2019 ground truth
    merged = meta_df.merge(gt_df, on="image", how="inner")
    merged["AKIEC"] = merged["AK"] + merged["SCC"]
    assert int(merged["UNK"].sum()) == 0
    cls_cols = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
    merged["class_7"] = merged[cls_cols].idxmax(axis=1)
    merged["has_lesion_id"] = merged["lesion_id"].notna() & (merged["lesion_id"].astype(str).str.strip() != "")
    merged["op_group"] = np.where(merged["has_lesion_id"], merged["lesion_id"].astype(str), "IMG::" + merged["image"].astype(str))

    split_sets = {}
    lesion_sets = {}
    audit_rows = []
    cls_rows = []
    for sname, spath in SPLIT_CSVS.items():
        ids = [ln.strip() for ln in spath.read_text(encoding="utf-8").splitlines() if ln.strip()]
        split_sets[sname] = set(ids)
        sub = merged[merged["image"].isin(split_sets[sname])]
        obs_lids = set(sub.loc[sub["has_lesion_id"], "lesion_id"].astype(str))
        lesion_sets[sname] = obs_lids
        missing_sing = int((~sub["has_lesion_id"]).sum())
        audit_rows.append({
            "split": sname,
            "n_images": len(sub),
            "observed_lesion_groups": len(obs_lids),
            "missing_id_singletons": missing_sing,
            "total_op_groups": len(obs_lids) + missing_sing,
            "age_missing_pct": round(100.0 * sub["age_approx"].isna().mean(), 2),
            "sex_missing_pct": round(100.0 * sub["sex"].isna().mean(), 2),
        })
        counts = sub["class_7"].value_counts().to_dict()
        cls_rows.append({"split": sname, **{c: int(counts.get(c, 0)) for c in cls_cols}})

    raw_audit_df = pd.DataFrame(audit_rows)
    class_split_df = pd.DataFrame(cls_rows)

    pairs = []
    names = list(SPLIT_CSVS.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            img_ov = len(split_sets[a] & split_sets[b])
            lid_ov = len(lesion_sets[a] & lesion_sets[b])
            assert img_ov == 0 and lid_ov == 0, f"Leakage detected between {a} and {b}!"
            pairs.append({"pair": f"{a} vs {b}", "image_id_overlap": img_ov, "observed_lesion_id_overlap": lid_ov})
    overlap_df = pd.DataFrame(pairs)
    print("FULL ROW-LEVEL AUDIT PASSED: 0 image_id overlaps and 0 lesion_id overlaps across all 6 split pairs.")
    display(raw_audit_df)
    display(overlap_df)
    display(class_split_df)
else:
    print("SKIPPED (REQUIRES_DATA): Raw ISIC 2019 CSVs or split index CSVs are not present in this clone.")
    print("To run row-level overlap auditing, acquire ISIC 2019 (CC-BY-NC 4.0) from:")
    print("  https://challenge.isic-archive.com/data/#2019")
    print("Expected placement:")
    print("  data/isic2019/raw/ISIC_2019_Training_Metadata.csv")
    print("  data/isic2019/raw/ISIC_2019_Training_GroundTruth.csv")
    print("  data/isic2019/splits/{train,val,cal,test}_indices.csv")

## 5. Partition & Operational Group Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Panel A: Images vs Operational Grouping Units from tracked split_metadata.json
sub_s = split_summary[split_summary["split"] != "Total ISIC 2019"]
x = np.arange(len(sub_s))
w = 0.36
axes[0].bar(x - w / 2, sub_s["images_N"], width=w, label="Images (N)", color="#2b6cb0")
axes[0].bar(x + w / 2, sub_s["groups_K"], width=w, label="Operational Groups (K)", color="#dd6b20")
axes[0].set_xticks(x)
axes[0].set_xticklabels(sub_s["split"])
axes[0].set_ylabel("Count")
axes[0].set_title("ISIC 2019 Partition Size vs. Operational Groups")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Panel B: Class distribution (live from raw CSVs if available, else parsed from tracked tab1_dataset_partition.tex)
cls_cols = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
if class_split_df is not None:
    totals = class_split_df[cls_cols].sum()
    src_note = "Verified live from ISIC_2019_Training_GroundTruth.csv"
else:
    import re
    tdir = find_tables_dir(REPO_ROOT)
    tab1_lines = (tdir / "tab1_dataset_partition.tex").read_text(encoding="utf-8").splitlines()
    tot_line = next(ln for ln in tab1_lines if "Total ISIC 2019" in ln)
    parts = [tok.strip() for tok in tot_line.replace("\\\\", "").split("&")]
    nums = [int(re.sub(r"[^0-9]", "", tok)) for tok in parts[3:10]]
    totals = pd.Series(nums, index=cls_cols)
    src_note = "Parsed from tracked supplementary/tables/tab1_dataset_partition.tex"

axes[1].bar(totals.index, totals.values, color="#2f855a")
axes[1].set_ylabel("Total Images (N = 25,331)")
axes[1].set_title("Seven-Class Benchmark Distribution (AK + SCC -> AKIEC)")
axes[1].grid(axis="y", alpha=0.3)
for i, v in enumerate(totals.values):
    axes[1].text(i, v + 200, f"{v:,}", ha="center", fontsize=8)

fig.suptitle(f"ISIC 2019 Provenance & Split Audit ({src_note})", fontsize=10, y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / "isic2019_split_audit.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Interpretation, Licensing & Limitations

- **Absence of `patient_id`:** Although zero pairwise `lesion_id` overlap holds across splits, ISIC 2019 does not supply a verified patient identifier; therefore **patient-level independence has not been established**, and multiple distinct lesions from the same individual could span splits.
- **Severe Class Imbalance:** `NV` (`12,875`, `50.8%`) dominates the benchmark, whereas `DF` (`239`, `0.9%`) and `VASC` (`253`, `1.0%`) yield very sparse calibration and test strata (`DF` test `n = 25`, `VASC` test `n = 19`).
- **Licensing & Attribution (`CC-BY-NC 4.0`):** Upstream ISIC 2019 images combine `BCN_20000`, `HAM10000`, and `MSK` under `CC-BY-NC 4.0`. Non-commercial restrictions propagate to derivative checkpoints and caches.

### Reproduction Information & References
- Seed: `42` (`split_hash = df52266ea5c2a4b1`)
- Output plot: `outputs/notebooks/nb01_data_split/isic2019_split_audit.png`
- Related docs: [Data README](../data/README.md) · [Data Acquisition & Attribution](../docs/DATA_ACQUISITION_AND_ATTRIBUTION.md) · [Data Provenance](../provenance/DATA_PROVENANCE.md) · [License Status](../LICENSE_STATUS.md)